# Phase 2 · Resolve pathogen–host pairs and acquire PubMed records

This phase expands the configured selectors into the saved pair list, builds pair-aware PubMed searches, then searches and fetches every selected pair. The acquisition is resumable: completed pair searches are checkpointed, fetched metadata is cached by PMID, and rerunning the cell continues unfinished pairs. Full virus selections can take hours; progress, rate, and ETA are logged as it runs.

In [ ]:
import sys
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('02_runs_and_pubmed', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

## Step 1 · Load configuration and taxonomy references

In [ ]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.runs import resolve_run, save_resolved_run
from pathogen_atlas_v4.pubmed import build_pair_query
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
LOG.info('Inputs loaded: hosts=%d pathogens=%d config=%s', len(hosts), len(pathogens), CONFIG['run']['run_id'])
display(CONFIG['run'])

## Step 2 · Resolve the exact pair set

In [ ]:
host_groups = {key: group.host_id.astype(str).tolist() for key, group in hosts.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in pathogens.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(hosts.host_id.astype(str)),
                       known_pathogen_ids=set(pathogens.pathogen_id.astype(str)),
                       host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
RESOLVED_PATH = ROOT / 'data/v4/runs' / resolved.run_id / 'resolved_config.json'
save_resolved_run(CONFIG, resolved, RESOLVED_PATH)
print('Run:', resolved.run_id, '| generation rule:', resolved.pair_generation)
print('Hosts:', len(resolved.host_ids), '| pathogens:', len(resolved.pathogen_ids), '| pairs:', len(resolved.pairs))
print('Resolved configuration:', RESOLVED_PATH)
LOG.info('Resolved and saved run: hosts=%d pathogens=%d pairs=%d hash=%s', len(resolved.host_ids), len(resolved.pathogen_ids), len(resolved.pairs), resolved.resolved_config_hash)

## Step 3 · Build and inspect pair-aware query text

In [ ]:
host_rows = hosts.set_index('host_id')
pathogen_rows = pathogens.set_index('pathogen_id')
PREVIEW_ROWS = []
for pathogen_id, host_id in resolved.pairs[:10]:
    pathogen = pathogen_rows.loc[pathogen_id]
    host = host_rows.loc[host_id]
    host_aliases = host.get('aliases', [])
    if hasattr(host_aliases, 'tolist'):
        host_aliases = host_aliases.tolist()
    query = build_pair_query([pathogen.accepted_name], [host.accepted_name, *(host_aliases or [])],
                             strategy_version=CONFIG['acquisition']['query_strategy_version'])
    PREVIEW_ROWS.append({'pathogen_id': pathogen_id, 'pathogen': pathogen.accepted_name,
                         'host_id': host_id, 'host': host.accepted_name, **query})
display(pd.DataFrame(PREVIEW_ROWS))
LOG.info('Generated %d preview queries; full acquisition covers all %d pairs with strategy %s', len(PREVIEW_ROWS), len(resolved.pairs), CONFIG['acquisition']['query_strategy_version'])


## Step 4 · Preflight counts and acquisition readiness

In [ ]:

PAIR_COUNT = len(resolved.pairs)
UNIQUE_PATHOGENS = len(resolved.pathogen_ids)
UNIQUE_HOSTS = len(resolved.host_ids)
ALIAS_PATH = TAXONOMY_ROOT / 'pathogen_aliases.parquet'
ALIASES = pd.read_parquet(ALIAS_PATH, columns=['pathogen_id', 'name']) if ALIAS_PATH.exists() else pd.DataFrame(columns=['pathogen_id', 'name'])
CACHE_DIR = ROOT / 'data/v4/runs' / resolved.run_id / 'literature' / resolved.resolved_config_hash
CACHE_DIR.mkdir(parents=True, exist_ok=True)
preflight = {'pairs_to_search': PAIR_COUNT, 'distinct_pathogens': UNIQUE_PATHOGENS, 'distinct_hosts': UNIQUE_HOSTS,
             'query_strategy_version': CONFIG['acquisition']['query_strategy_version'],
             'pair_checkpoint': str(CACHE_DIR / 'pair_search_checkpoints.jsonl'),
             'per_pair_queries': True, 'sampling_limit': None, 'network_calls_made': 0}
display(preflight)
LOG.info('PubMed preflight: pairs=%d pathogens=%d hosts=%d cache=%s aliases=%d', PAIR_COUNT, UNIQUE_PATHOGENS, UNIQUE_HOSTS, CACHE_DIR, len(ALIASES))


## Step 5 · Search every selected pair and fetch every result

The client uses NCBI E-utilities with contact identification, rate limiting, retries, per-pair checkpoints, and PMID-deduplicated article metadata. No pair or PMID sampling cap is applied. Search result sets above PubMed’s 10,000-record UID retrieval ceiling are segmented by publication date; a segment that still cannot be retrieved completely raises an error instead of silently truncating.

In [ ]:
import os
from pathogen_atlas_v4.pubmed_client import acquire_pubmed_pairs

NCBI_EMAIL = os.environ.get('NCBI_EMAIL') or input('NCBI contact email (saved only in your local environment): ').strip()
NCBI_API_KEY = os.environ.get('NCBI_API_KEY') or None
LOG.info('Starting full PubMed acquisition: pairs=%d api_key_configured=%s cache=%s', len(resolved.pairs), bool(NCBI_API_KEY), CACHE_DIR)
acquisition = acquire_pubmed_pairs(
    resolved.pairs, pathogens, hosts, ALIASES, CACHE_DIR,
    query_version=CONFIG['acquisition']['query_strategy_version'],
    email=NCBI_EMAIL, api_key=NCBI_API_KEY, logger=LOG,
)
display(acquisition)
LOG.info('PubMed acquisition complete: %s', acquisition)


## Outputs and resume behavior

The run-specific cache contains append-only pair checkpoints and title metadata. Materialized tables are written under `tables/`: `pair_searches.parquet`, `pathogen_host_article_matches.parquet`, and `articles.parquet`. The title and PMID are retained; abstracts can be opened directly from PubMed when needed. If interrupted, rerun this phase: completed pairs are skipped and the same resolved-configuration hash selects the same cache directory.